In [ ]:
# This notebook needs to work with the prosculpt env.


In [ ]:
import pandas as pd
import os
import shutil
import glob
import pathlib
import numpy as np
def dummy_npwarn_decorator_factory(): #This fixes an incompatibility with seaborn and numpy 2.0
  def npwarn_decorator(x):
    return x
  return npwarn_decorator
np._no_nep50_warning = getattr(np, '_no_nep50_warning', dummy_npwarn_decorator_factory)

import seaborn as sns

In [ ]:
# Set requirements for filtering

plddt_threshold = 80    
plddt_sculpted_threshold = 83
rmsd_threshold = 3

rmsd_fixedchains_threshold = 5
rmsd_motif_threshold = 5
rmsd_sculpted_threshold = 3

folder = "" #output folder of the prosculpt job
folders = glob.glob(folder + "/*")
print(folders)

In [ ]:
# Select best pdbs based on different criteria

good_results_dict = {}
copy_files = True  # Copy the files? Usually this is what you want

for subfolder in folders:

    if os.path.exists(os.path.join(subfolder, "best_pdbs")):
        shutil.rmtree(os.path.join(subfolder, "best_pdbs"))
    if os.path.exists(os.path.join(subfolder, "final_output_best.csv")):
        os.remove(os.path.join(subfolder, "final_output_best.csv"))

    if os.path.exists(os.path.join(subfolder, "final_output.csv")):
        
        df = pd.read_csv(os.path.join(subfolder, "final_output.csv"))
        # print(df['RMSD_sculpted'])
        # df['RMSD_sculpted'] = pd.to_numeric(df['RMSD_sculpted'])
        # best_df=df[(df["RMSD_sculpted"] <= rmsd_sculpted_threshold) & (df["plddt"] >= plddt_threshold)] #select best based on thresholds
        best_df = df[
            (df["RMSD"] <= rmsd_threshold)
            & (df["RMSD_sculpted"] <= rmsd_sculpted_threshold)
            & (df["plddt"] >= plddt_threshold)
            & (df["plddt_sculpted"] >= plddt_sculpted_threshold)
        ]  # select best based on thresholds
        # best_df=df[(df["RMSD_sculpted"] <= rmsd_sculpted_threshold) & (df["plddt_sculpted"] >= plddt_sculpted_threshold)] #select best based on thresholds
        # best_df=df[(df["RMSD_sculpted"] <= rmsd_sculpted_threshold) & (df["plddt"] >= plddt_threshold)& (df["plddt_sculpted"] >= plddt_sculpted_threshold)] #select best based on thresholds
        # best_df=df[(df["RMSD_motif"] <= rmsd_motif_threshold) & (df["plddt_sculpted"] >= plddt_sculpted_threshold) & (df["RMSD_fixed_chains"] <= rmsd_fixedchains_threshold) & (df["RMSD_sculpted"] <= rmsd_sculpted_threshold)]

        # best_df['id']= df.apply(lambda row: f"{pathlib.Path(row['path_rfdiff']).stem[1:]}.{row['link_lenght']:.{0}f}.{row['plddt_sculpted']:.{0}f}.{row['RMSD_sculpted']:.{0}f}.{row['pae']:.{0}f}", axis=1)
        # duplicates = best_df['id'].duplicated()
        # print('Number of duplicated ids: ', sum(duplicates))
        # print(best_df)

        # best_df=df[(df["plddt"] >= plddt_threshold)] #Only plddt
        # best_df=df[(df["RMSD_sculpted"] <= rmsd_sculpted_threshold)] #Only rmsd sculpted
        # best_df=df[(df["pae"] <= 8)] #Only rmsd sculpted
        dir_best_pdbs = os.path.join(subfolder, "best_pdbs")

        if copy_files:
            os.makedirs(
                dir_best_pdbs, exist_ok=True
            )  # directory is created even if some or all of the intermediate directories in the path do not exist
            best_folder = os.path.join(subfolder, "best_pdbs")
            for index, row in best_df.iterrows():
                file = str(row["model_path"])
                shutil.copy(file, os.path.join(best_folder, pathlib.Path(file).name))

                # file=file.replace('/home/folivieri/prosculpt/Fibers_Rollers_rigid_fusion/output/','/home/folivieri/prosculpt/Fibers_Rollers_rigid_fusion/99_2024-04-22/')
                # id=row["id"]
                # id_filename=id+'.pdb'
                # shutil.copy(file,os.path.join(best_folder,str(id_filename)))
        best_df = best_df.replace("final_pdbs", "best_pdbs", regex=True)

        best_df.to_csv(
            f'{os.path.join(subfolder, "final_output_best.csv")}', index=False
        )
        good_results_dict[pathlib.Path(subfolder).name] = len(best_df)

        print(subfolder, len(best_df))

In [ ]:
# Display number of results per task
good_results_df = pd.DataFrame.from_dict(good_results_dict, "index")
pd.set_option("display.max_rows", None)
display(good_results_df.sort_values(0, ascending=False))

In [ ]:
# Pool all pdbs that were selected above for each task into one folder

pdbs = glob.glob(os.path.abspath(folder) + "/*/best_pdbs/*")
print(pdbs)

pooled_folder = os.path.join(folder, "pooled_best_pdbs")
os.makedirs(pooled_folder, exist_ok=True)

for pdb in pdbs:
    shutil.copy(pdb, pooled_folder)

best_csvs = glob.glob(folder + "/*/final_output_best.csv")

df_list = []
for csv_file in best_csvs:
    df = pd.read_csv(csv_file)
    df_list.append(df)
big_df = pd.concat(df_list, ignore_index=True)


def get_id_from_path(model_path):
    return os.path.basename(model_path)


big_df["file_name"] = big_df["model_path"].apply(get_id_from_path)
big_df.to_csv(os.path.join(pooled_folder, "prosculpt_metrics.csv"), index=False)

In [ ]:
#Rosetta options


use_backbone_relax=True

resnums=0 #Only matters if using split
split=False




#Calculated parameters
chain1="A" #Don't touch, usually 
chain2="B"
print(pooled_folder)
output_name="rosetta_metrics"
pdbs_folder=pooled_folder
if split:
    option="<"
else:
    option=""


In [ ]:
#Rosetta scripts
Rosetta_script_string_bb_relax=f"""
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="sfxn_clean" weights="beta_nov16_cart" />
        </SCOREFXNS>

        <RESIDUE_SELECTORS>
            <Chain name="chain1" chains="{chain1}" />
            <Chain name="chain2" chains="{chain2}" />
            <Chain name="chainA" chains="A" />
            <Chain name="chainB" chains="B" />
            <ScoreTermValueBased name="clashing_res" score_type="fa_rep" score_fxn="sfxn"  lower_threshold="3" upper_threshold="99999" />
            <Index name="chain_sep" resnums="{resnums}"/>
            <Index name="chain_sep_from_end" resnums="{resnums}" reverse="true"/>

        </RESIDUE_SELECTORS>
        <TASKOPERATIONS>
            <IncludeCurrent name="ic" /> //includes input pdbs rotamers
            <LimitAromaChi2 name="limitaro" chi2max="110" chi2min="70" include_trp="1" /> //disallow extreme aromatic rotamers
            <ExtraRotamersGeneric name="ex1_ex2" ex1="1" ex2="1" /> //use ex1 ex2 rotamers
            <RestrictToRepacking name="repack_only" />  //for minimize/repack
        </TASKOPERATIONS>

        <FILTERS>
            <NetCharge name="chargeA" chain="1" confidence="0" />
            <NetCharge name="charge_total" chain="0" confidence="0" />
            <ShapeComplementarity name="sc2" min_sc="0.6" verbose="1" quick="0" residue_selector1="chainA" residue_selector2="chainB" write_int_area="1" write_median_dist="1" confidence="0" />
            <ExposedHydrophobics name="exposed_hydrop" sasa_cutoff="20" threshold="0" confidence="0"/>
            <ContactMolecularSurface name="cms" distance_weight="0.5" use_rosetta_radii="true" apolar_target="0"
            target_selector="chain1" binder_selector="chain2" confidence="0" />

            <BuriedUnsatHbonds name="vbuns"  report_all_heavy_atom_unsats="true" scorefxn="sfxn_clean" ignore_surface_res="false" print_out_info_to_pdb="true" atomic_depth_selection="5" burial_cutoff="1000" residue_surface_cutoff="42.5" dalphaball_sasa="0" confidence="0"  only_interface="false"  />
            <BuriedUnsatHbonds2 name="vbuns2"  scorefxn="sfxn_clean" confidence="0" cutoff="5"/>

            <BuriedUnsatHbonds name="sbuns" report_all_heavy_atom_unsats="true" scorefxn="sfxn_clean" cutoff="4" residue_surface_cutoff="42.5" ignore_surface_res="false" print_out_info_to_pdb="true" dalphaball_sasa="0" probe_radius="1.1" atomic_depth_selection="5.5" atomic_depth_deeper_than="false" only_interface="false" confidence="0" />

        </FILTERS>

        <MOVERS>
            <SavePoseMover name="save_current" restore_pose="0" reference_name="original" />
            <DeleteRegionMover name="split_chains" residue_selector="chain_sep" rechain="1" />
            <SwitchChainOrder name="take_two_chains" chain_name="{chain1},{chain2}"/> #Or whatever chain you need to get one neighbor
            <MinMover name="minimize_sc_all_cart" scorefxn="sfxn_clean" bb="0" chi="1" cartesian="true" />
            <MinMover name="minimize_bb_all_cart" scorefxn="sfxn_clean" bb="1" chi="1" cartesian="true" />
            <InterfaceAnalyzerMover name="analyze_interface" scorefxn="sfxn_clean"
            packstat="1" interface_sc="1" use_jobname="1"
            jump="1" scorefile_reporting_prefix="IA" />
            <DumpPdb name="dump_pose" fname="dump.pdb" />
            <ddG name="ddG_no_repack" translate_by="1000" scorefxn="sfxn_clean"  task_operations="repack_only,ic,ex1_ex2" relax_mover="minimize_sc_all_cart"
                repack_bound="0"
                relax_bound="0"
                repack_unbound="0"
                relax_unbound="1"
            jump="1"
            dump_pdbs="0"   />
        </MOVERS>
        <SIMPLE_METRICS>
            <RMSDMetric
                name        = "rmsd_to_original"
                reference_name = "original"
                residue_selector = "chainB"
                residue_selector_ref = "chain2"
                rmsd_type   = "rmsd_protein_bb_heavy"
            />
            <SapScoreMetric name="sap_score" />
            <SelectedResiduesPyMOLMetric name="clashing_res" residue_selector="clashing_res" custom_type="clashing_res" />
        </SIMPLE_METRICS>
        <PROTOCOLS>
            <Add mover_name="save_current" />
            {option}Add mover="split_chains"/>
            <Add mover="take_two_chains" />
            Add mover='dump_pose' />
            <Add mover="minimize_sc_all_cart" />
            <Add mover="minimize_bb_all_cart" />
            <Add mover="minimize_sc_all_cart" />
            <Add mover="analyze_interface" />
            <Add mover="ddG_no_repack" />
            <Add filter="chargeA" />
            <Add filter="charge_total" />
            <Add filter="exposed_hydrop" />
            <Add filter="cms" />
            <Add filter="vbuns" />
            <Add filter="vbuns2" />
            <Add filter="sbuns" />
            <Add metrics="sap_score" />
            <Add metrics="clashing_res" />
            <Add metrics="rmsd_to_original" />
            <Add filter="sc2" />
        </PROTOCOLS>
    </ROSETTASCRIPTS>

    """


#Original
Rosetta_script_string_no_bb_relax=f"""
    <ROSETTASCRIPTS>
        <SCOREFXNS>
            <ScoreFunction name="sfxn_clean" weights="beta_nov16" />
        </SCOREFXNS>

        <RESIDUE_SELECTORS>
            <Chain name="chain1" chains="{chain1}" />
            <Chain name="chain2" chains="{chain2}" />
            <Chain name="chainA" chains="A" />
            <Chain name="chainB" chains="B" />
            <ScoreTermValueBased name="clashing_res" score_type="fa_rep" score_fxn="sfxn"  lower_threshold="3" upper_threshold="99999" />
            <Index name="chain_sep" resnums="{resnums}"/>
            <Index name="chain_sep_from_end" resnums="{resnums}" reverse="true"/>

        </RESIDUE_SELECTORS>
        <TASKOPERATIONS>
            <IncludeCurrent name="ic" /> //includes input pdbs rotamers
            <LimitAromaChi2 name="limitaro" chi2max="110" chi2min="70" include_trp="1" /> //disallow extreme aromatic rotamers
            <ExtraRotamersGeneric name="ex1_ex2" ex1="1" ex2="1" /> //use ex1 ex2 rotamers
            <RestrictToRepacking name="repack_only" />  //for minimize/repack
        </TASKOPERATIONS>

        <FILTERS>
            <NetCharge name="chargeA" chain="1" confidence="0" />
            <NetCharge name="charge_total" chain="0" confidence="0" />
            <ShapeComplementarity name="sc2" min_sc="0.6" verbose="1" quick="0" residue_selector1="chainA" residue_selector2="chainB" write_int_area="1" write_median_dist="1" confidence="0" />
            <ExposedHydrophobics name="exposed_hydrop" sasa_cutoff="20" threshold="0" confidence="0"/>
            <ContactMolecularSurface name="cms" distance_weight="0.5" use_rosetta_radii="true" apolar_target="0"
            target_selector="chain1" binder_selector="chain2" confidence="0" />

            <BuriedUnsatHbonds name="vbuns"  report_all_heavy_atom_unsats="true" scorefxn="sfxn_clean" ignore_surface_res="false" print_out_info_to_pdb="true" atomic_depth_selection="5" burial_cutoff="1000" residue_surface_cutoff="42.5" dalphaball_sasa="0" confidence="0"  only_interface="false"  />
            <BuriedUnsatHbonds2 name="vbuns2"  scorefxn="sfxn_clean" confidence="0" cutoff="5"/>

            <BuriedUnsatHbonds name="sbuns" report_all_heavy_atom_unsats="true" scorefxn="sfxn_clean" cutoff="4" residue_surface_cutoff="42.5" ignore_surface_res="false" print_out_info_to_pdb="true" dalphaball_sasa="0" probe_radius="1.1" atomic_depth_selection="5.5" atomic_depth_deeper_than="false" only_interface="false" confidence="0" />
            
        </FILTERS>

        <MOVERS>
            <DeleteRegionMover name="split_chains" residue_selector="chain_sep" rechain="1" />
            <SwitchChainOrder name="take_two_chains" chain_name="{chain1},{chain2}"/> #Or whatever chain you need to get one neighbor
            <MinMover name="minimize_sc_all" scorefxn="sfxn_clean" bb="0" chi="1" />
            <InterfaceAnalyzerMover name="analyze_interface" scorefxn="sfxn_clean"
            packstat="1" interface_sc="1" use_jobname="1"
            jump="1" scorefile_reporting_prefix="IA" />
            <DumpPdb name="dump_pose" fname="dump.pdb" />
            <ddG name="ddG_no_repack" translate_by="1000" scorefxn="sfxn_clean"  task_operations="repack_only,ic,ex1_ex2" relax_mover="minimize_sc_all"
                repack_bound="0"
                relax_bound="0"
                repack_unbound="0"
                relax_unbound="1"
            jump="1"
            dump_pdbs="0"   />
        </MOVERS>
        <SIMPLE_METRICS>
            <SapScoreMetric name="sap_score" />
            <SelectedResiduesPyMOLMetric name="clashing_res" residue_selector="clashing_res" custom_type="clashing_res" />
        </SIMPLE_METRICS>
        <PROTOCOLS>
            
            {option}Add mover="split_chains"/>
            <Add mover="take_two_chains" />
            Add mover='dump_pose' />
            <Add mover="minimize_sc_all" />
            <Add mover="analyze_interface" />
            <Add mover="ddG_no_repack" />
            <Add filter="chargeA" />
            <Add filter="charge_total" />
            <Add filter="exposed_hydrop" />
            <Add filter="cms" />
            <Add filter="vbuns" />
            <Add filter="vbuns2" />
            <Add filter="sbuns" />
            <Add metrics="sap_score" />
            <Add metrics="clashing_res" />
            <Add filter="sc2" />
        </PROTOCOLS>
    </ROSETTASCRIPTS>

    """

In [ ]:
#Rosetta script running Code

if use_backbone_relax:
    Rosetta_script_string = Rosetta_script_string_bb_relax
else:
    Rosetta_script_string = Rosetta_script_string_no_bb_relax

code=f"""
import os
import pandas as pd
import pyrosetta
from Bio import SeqIO
import argparse
from jug import TaskGenerator
import glob
import pathlib
import shutil

parser = argparse.ArgumentParser(description="Run rosetta metrics.")
parser.add_argument("in_name", help="Input folder path")
parser.add_argument("out_name", help="Output file name")
# parser.add_argument('-s', '--split',action='store_true')
args = parser.parse_args()

in_folder = args.in_name
outfile = args.out_name

os.makedirs(os.path.join(in_folder, f"{output_name}_minimized"), exist_ok=True)

@TaskGenerator
def calculate_metrics(pdb):
    pose = pyrosetta.pose_from_pdb(os.path.join(in_folder, pdb))
    protocol = (
        pyrosetta.rosetta.protocols.rosetta_scripts.XmlObjects()
        .create_from_string(\"\"\"
            {Rosetta_script_string}
            \"\"\"
        )
        .get_mover("ParsedProtocol")
    )
    protocol.apply(pose)
    extra_dict = dict(pose.scores)
    job_pairs = pyrosetta.rosetta.protocols.jd2.get_string_real_pairs_from_current_job()
    pyrosetta.dump_pdb(pose, os.path.join(in_folder, f"{output_name}_minimized", pdb))
    # pyrosetta.dump_pdb(pose, os.path.join(in_folder, f"{output_name}_minimized_assemble", pdb))
    if csv_found:
        extra_dict.update(prosculpt_dict[pdb])

    for name in job_pairs:
        if (
            not name in extra_dict
        ):  # for some strange reason things may be written to both dicts
            extra_dict[name] = job_pairs[name]

    return extra_dict


@TaskGenerator
def save_output(st):
    df = pd.DataFrame.from_dict(st, "index")
    df.index.name = "ID"
    pd.DataFrame.to_csv(df, os.path.join(in_folder, outfile+".csv"))


pyrosetta.init(
    "-mute all -ignore_unrecognized_res -ignore_zero_occupancy -corrections::beta_nov16 true"
)


os.chdir(in_folder)

try:
    prosculpt_dict = (
        pd.read_csv("prosculpt_metrics.csv")
        .set_index("file_name")
        .to_dict(orient="index")
    )
    csv_found = True
    print("CSV found")
except:
    csv_found = False

pdb_list = glob.glob("*.pdb")
st = {{}}
for pdb in pdb_list:
    st[pdb] = calculate_metrics(pdb)

print(st)
save_output(st)
"""
with open(os.path.join(pdbs_folder, f"{output_name}.py"), "w") as f:
    f.write(code)


In [ ]:
# Write rosetta slurm file
slurm_file=f"""#!/bin/sh
#SBATCH --partition=amd,intel
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --output logs/slurm-%A_%a_metrics.out
#SBATCH --error logs/slurm-%A_%a_metrics.err
#SBATCH -J r_metrics

jug status {pdbs_folder}/{output_name}.py {pdbs_folder} {output_name}
jug execute {pdbs_folder}/{output_name}.py {pdbs_folder} {output_name}
"""
with open(os.path.join(pdbs_folder, f"{output_name}.slurm"), "w") as slurm_f:
    slurm_f.write(slurm_file)

In [ ]:
#Run rosetta metrics on pooled best structures


import glob

print(pdbs_folder)
if len(glob.glob(pdbs_folder+"/*.pdb")) != 0:
    print(f"found {len(glob.glob(pdbs_folder+"/*.pdb"))} pdbs")
    !sbatch --array=1-{int(len(glob.glob(pdbs_folder+"/*.pdb"))/2)} {os.path.join(pdbs_folder, output_name+".slurm")}




In [ ]:
# Wait until the tasks finish
# remove the jug data (this is important. it interferes next time you want to run it)
shutil.rmtree(f"{pdbs_folder}/{output_name}.jugdata/")


In [ ]:
#Pool multiple best_pdbs (optional, if you want to pool the best pdbs from multiple jobs)
#TODO Fix this

all_outputs_folder = folder
all_pooled_folder = os.path.join(all_outputs_folder, "all_pooled_best_pdbs")
os.makedirs(all_pooled_folder, exist_ok=True)
print(all_outputs_folder)
all_dfs = []

for folder in glob.glob(all_outputs_folder):
    print(f"Checking {folder}")
    
    rosetta_csv = os.path.join(folder, "rosetta.csv")
    if os.path.isfile(rosetta_csv):
        #print("found rosetta.csv in", folder)
        df = pd.read_csv(rosetta_csv)
        
        # copy pdbs
        pdbs = glob.glob(folder + "/*.pdb")
        print(f"Found {len(pdbs)} pdbs in {folder}")
        for pdb in pdbs:
            
            # keep filename only
            pdb_id = os.path.basename(pdb)
            final_path = os.path.join(all_pooled_folder, pdb_id)
            shutil.copy(pdb, final_path)
            
            # match row in df with this pdb_id if column exists
            df["final_model_path"] = final_path
        
        all_dfs.append(df)

# concatenate everything
if all_dfs:
    final_df = pd.concat(all_dfs, ignore_index=True)
    final_csv_path = os.path.join(all_pooled_folder, "all_rosetta.csv")
    final_df.to_csv(final_csv_path, index=False)
    print(f"Saved concatenated CSV to {final_csv_path}")
else:
    print("No rosetta.csv files found")

In [ ]:
# Generate a Fasta

from Bio import SeqIO

pdb_list = glob.glob(pooled_folder + "/*.pdb")

with open(os.path.join(pooled_folder, "sequences.fasta"), "w+") as f:
    f.write("# --amber\n")
    for pdb in pdb_list:
        f.write(f">{pdb}\n")
        print(pdb)
        seq = ""
        for record in SeqIO.parse(os.path.join(pooled_folder, pdb), "pdb-atom"):
            seq += str(record.seq)
            seq += ":"

        seq = seq[:-1]  # delete last ":"
        seq += "\n"
        f.write(seq)


chains = [0]
with open(os.path.join(pooled_folder, "sequences_monomer.fasta"), "w+") as f:
    f.write("# --amber\n")
    for pdb in pdb_list:
        f.write(f">{pdb}\n")
        print(pdb)
        seq = ""
        chain_n = 0
        for record in SeqIO.parse(os.path.join(pooled_folder, pdb), "pdb-atom"):
            if chain_n in chains:
                seq += str(record.seq)
                seq += ":"
            chain_n += 1

        seq = seq[:-1]  # delete last ":"
        seq += "\n"
        f.write(seq)

In [ ]:
pooled_folder = os.path.join(folder, "pooled_best_pdbs")
#pooled_folder = os.path.join(folder)
prosculpt_csv_path = os.path.join(pooled_folder, "prosculpt_metrics.csv")
rosetta_csv_path = os.path.join(pooled_folder, f"{output_name}.csv")


def get_monomer_path(af2_path):
    p = pathlib.Path(af2_path)
    pm = p.parent / "monomers" / ("monomer_" + p.name)
    # pm.exists()
    return str(pm)


df = pd.read_csv(rosetta_csv_path, index_col="ID")
df_prosculpt = pd.read_csv(prosculpt_csv_path)
filtered_path = os.path.join(pooled_folder, "filtered")
# df.drop(columns=['Unnamed: 0'], inplace=True)

df["file_name"] = df.index
cols_to_add = [
    c for c in df_prosculpt.columns
    if c not in df.columns or c == "file_name"
]
df = pd.merge(
    df,
    df_prosculpt[cols_to_add],
    how="outer",
    on="file_name"
)
new_order = ["file_name"] + [col for col in df.columns if col != "file_name"]
df = df[new_order]


def get_new_path(filtered_path, file_name):
    return os.path.join(filtered_path, file_name)


df["model_path"] = df.apply(
    lambda x: get_new_path(filtered_path, x["file_name"]), axis=1
)

print(df)
interesting_cols = [
    "file_name",
    "plddt",
    "plddt_sculpted",
    "pae",
    "chargeA",
    "sap_score",
    "ddg",
    "sc2",
    "vbuns",
    "sbuns",
    "RMSD",
    "RMSD_fixed_chains",
    "RMSD_motif",
]

if pathlib.Path(filtered_path).exists() and pathlib.Path(filtered_path).is_dir():
    shutil.rmtree(pathlib.Path(filtered_path))
os.makedirs(filtered_path, exist_ok=True)

#dfq = df.query("plddt>90").sort_values(by="plddt", ascending=False)
dfq = df.query('plddt_sculpted>90').sort_values(by='plddt_sculpted', ascending=False)
dfq = dfq.query("ddg<0").sort_values(by="ddg", ascending=False)
dfq = dfq.query("RMSD_sculpted<4").sort_values(by="RMSD_sculpted", ascending=False)
# dfq = dfq.query('sc2>0.595').sort_values(by='sc2', ascending=False)
# dfq = dfq.query('vbuns<10').sort_values(by='vbuns', ascending=False)
# dfq = dfq.query('chargeA<-2').sort_values(by='chargeA', ascending=False)
#dfq = dfq.query("RMSD_motif<=5").sort_values(by="RMSD_motif", ascending=False)

# dfq = dfq.query('ddg>-70').sort_values(by='ddg', ascending=False)

#print(dfq[interesting_cols])

for index, row in dfq.iterrows():
    shutil.copy(
        os.path.join(pooled_folder, "minimized", row["file_name"]), filtered_path
    )

dfq.to_csv(os.path.join(filtered_path, "rosetta_metrics_selected.csv"))
dfs = dfq[["plddt_sculpted", "sc2", "ddg", "RMSD_sculpted"]]
sns.pairplot(dfs)